# CoherenceBench-IN — GPU Standalone Evaluation

**Self-contained. Run on any CUDA GPU PC with VS Code.**

### What this notebook does (fully automated)
1. Installs all required packages
2. Downloads the benchmark data from GitHub (~30 MB)
3. Runs 3 LLMs with 4-bit quantization on your GPU
4. Saves results to `cb_results/` and zips them for easy transfer

### How to use
1. Copy this file to the GPU PC (USB / email / GitHub clone)
2. Open in VS Code → select a Python kernel
3. **Run All** — takes ~1–3 hours depending on GPU
4. Copy `cb_results.zip` back and share

### Requirements
- Python 3.9 or higher
- CUDA GPU (any VRAM ≥ 6 GB — uses 4-bit quantization)
- Internet access (to download models from HuggingFace)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — Install packages  (runs once, ~2 min)
# ═══════════════════════════════════════════════════════════
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

print('Installing packages...')

# PyTorch with CUDA — detect version automatically
import subprocess
try:
    nvcc_out = subprocess.check_output(['nvcc', '--version'], text=True)
    if '12.1' in nvcc_out or '12.2' in nvcc_out or '12.3' in nvcc_out or '12.4' in nvcc_out:
        cuda_tag = 'cu121'
    elif '11.8' in nvcc_out or '11.7' in nvcc_out:
        cuda_tag = 'cu118'
    else:
        cuda_tag = 'cu121'   # safe default for most modern servers
except Exception:
    cuda_tag = 'cu121'

print(f'  CUDA tag: {cuda_tag}')
pip('torch', 'torchvision', '--index-url', f'https://download.pytorch.org/whl/{cuda_tag}')

# HF stack + quantization
pip('transformers>=4.45.0', 'accelerate>=0.34.0', 'bitsandbytes>=0.43.0',
    'sentencepiece', 'protobuf', 'safetensors', 'huggingface_hub>=0.24.0')

# Utilities
pip('tiktoken', 'tqdm', 'pandas', 'numpy', 'requests')

print('\n✅ All packages installed.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Download benchmark data from GitHub
# ═══════════════════════════════════════════════════════════
import requests, json, zipfile, io, os
from pathlib import Path
from tqdm.auto import tqdm

REPO_API  = 'https://api.github.com/repos/jeeth-kataria/coherencebench-in'
RAW_BASE  = 'https://raw.githubusercontent.com/jeeth-kataria/coherencebench-in/main'
DATA_DIR  = Path('cb_data/benchmark/english')
DATA_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = Path('cb_results')
RESULTS_DIR.mkdir(exist_ok=True)

index_path = DATA_DIR / 'index.jsonl'

if index_path.exists():
    existing = sum(1 for _ in open(index_path))
    print(f'Benchmark already downloaded ({existing} instances in index.jsonl).')
else:
    print('Downloading benchmark index...')
    r = requests.get(f'{RAW_BASE}/data/benchmark/english/index.jsonl')
    r.raise_for_status()
    index_path.write_bytes(r.content)
    instances_meta = [json.loads(l) for l in open(index_path)]
    print(f'  Index: {len(instances_meta)} instances')

    print('Downloading instance JSON files (this takes a few minutes)...')
    instances_meta = [json.loads(l) for l in open(index_path)]
    for inst in tqdm(instances_meta, desc='Downloading instances'):
        dest = DATA_DIR / f"{inst['id']}.json"
        if dest.exists():
            continue
        url = f"{RAW_BASE}/data/benchmark/english/{inst['id']}.json"
        resp = requests.get(url)
        resp.raise_for_status()
        dest.write_bytes(resp.content)

print(f'\n✅ Benchmark data ready at {DATA_DIR.resolve()}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — Configuration & GPU check
# ═══════════════════════════════════════════════════════════
import torch, re, time
import pandas as pd
import numpy as np

if not torch.cuda.is_available():
    raise RuntimeError(
        'No CUDA GPU detected.\n'
        'Make sure you are on a GPU node and PyTorch is installed with CUDA support.\n'
        'Check with: nvidia-smi'
    )

GPU_NAME  = torch.cuda.get_device_name(0)
GPU_VRAM  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU : {GPU_NAME}')
print(f'VRAM: {GPU_VRAM:.1f} GB')
print(f'PyTorch: {torch.__version__}  |  CUDA: {torch.version.cuda}')

MAX_CTX_TOKENS = 16_000
MAX_NEW_TOKENS = 15

SYSTEM_PROMPT = (
    'You are a precise document analyst. '
    'Read the document carefully and answer the question about its internal consistency. '
    'Your answer must be exactly ONE word: either CONSISTENT or INCONSISTENT. '
    'Do not explain. Do not add punctuation. Output only the single word.'
)

print('\n✅ Configuration ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Load benchmark + helper functions
# ═══════════════════════════════════════════════════════════
import tiktoken
enc = tiktoken.get_encoding('cl100k_base')

instances = [json.loads(l) for l in open(DATA_DIR / 'index.jsonl')]
df_bench  = pd.DataFrame(instances)
print(f'Loaded {len(instances)} instances')
print(df_bench.groupby(['dimension', 'answer']).size().unstack(fill_value=0))


def load_text(inst_id: str) -> str:
    with open(DATA_DIR / f'{inst_id}.json') as f:
        return json.load(f)['text']


def build_prompt(text: str, question: str) -> str:
    q_tok   = len(enc.encode(question))
    budget  = MAX_CTX_TOKENS - q_tok - 512
    d_tok   = enc.encode(text)
    if len(d_tok) > budget:
        text = enc.decode(d_tok[:budget]) + ' [...document truncated...]'
    return f'Document:\n{text}\n\n---\n{question}'


def parse_answer(response: str) -> str:
    if re.search(r'\bINCONSISTENT\b', response, re.IGNORECASE): return 'INCONSISTENT'
    if re.search(r'\bCONSISTENT\b',   response, re.IGNORECASE): return 'CONSISTENT'
    return 'UNPARSEABLE'


def load_done_ids(results_file: Path) -> set:
    if not results_file.exists(): return set()
    return {json.loads(l)['id'] for l in open(results_file)}


print('\n✅ Benchmark loaded.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — GPU inference function (4-bit, bitsandbytes)
# ═══════════════════════════════════════════════════════════
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


def run_model(hf_id: str, run_name: str) -> pd.DataFrame:
    """
    Load model in 4-bit on CUDA, evaluate all 584 instances.
    Checkpointed — safe to interrupt and resume.
    """
    results_file = RESULTS_DIR / f'{run_name}_results.jsonl'
    done_ids     = load_done_ids(results_file)
    remaining    = [i for i in instances if i['id'] not in done_ids]

    if not remaining:
        print(f'  {run_name}: already complete ({len(instances)} instances).')
    else:
        if done_ids:
            print(f'  Resuming {run_name} — {len(done_ids)} done, {len(remaining)} remaining.')

        print(f'  Loading tokenizer: {hf_id}')
        tokenizer = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=True)
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token_id = tokenizer.eos_token_id

        quant_cfg = BitsAndBytesConfig(
            load_in_4bit              = True,
            bnb_4bit_compute_dtype    = torch.float16,
            bnb_4bit_quant_type       = 'nf4',
            bnb_4bit_use_double_quant = True,
        )

        print(f'  Loading model in 4-bit on GPU...')
        model = AutoModelForCausalLM.from_pretrained(
            hf_id,
            quantization_config = quant_cfg,
            device_map          = 'auto',
            trust_remote_code   = True,
        )
        model.eval()
        used_gb = torch.cuda.memory_allocated(0) / 1024**3
        print(f'  Model loaded. VRAM used: {used_gb:.1f} GB')
        print(f'  Evaluating {len(remaining)} instances...')

        unparseable = skipped = 0
        with open(results_file, 'a') as out_f:
            for inst in tqdm(remaining, desc=run_name, unit='inst', dynamic_ncols=True):
                try:
                    text     = load_text(inst['id'])
                    user_msg = build_prompt(text, inst['question'])
                    messages = [
                        {'role': 'system', 'content': SYSTEM_PROMPT},
                        {'role': 'user',   'content': user_msg},
                    ]
                    formatted = tokenizer.apply_chat_template(
                        messages, tokenize=False, add_generation_prompt=True
                    )
                    inputs = tokenizer(formatted, return_tensors='pt').to('cuda')

                    # Clamp to max context
                    if inputs['input_ids'].shape[1] > MAX_CTX_TOKENS + 512:
                        inputs['input_ids']      = inputs['input_ids'][:, -(MAX_CTX_TOKENS+512):]
                        inputs['attention_mask'] = inputs['attention_mask'][:, -(MAX_CTX_TOKENS+512):]

                    with torch.no_grad():
                        out = model.generate(
                            **inputs,
                            max_new_tokens = MAX_NEW_TOKENS,
                            do_sample      = False,
                            pad_token_id   = tokenizer.pad_token_id,
                        )

                    new_tok  = out[0][inputs['input_ids'].shape[1]:]
                    response = tokenizer.decode(new_tok, skip_special_tokens=True).strip()
                    pred     = parse_answer(response)
                    correct  = (pred == inst['answer'])
                    if pred == 'UNPARSEABLE': unparseable += 1

                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    response, pred, correct = 'OOM', 'UNPARSEABLE', False
                    skipped += 1

                except KeyboardInterrupt:
                    print('\n  Interrupted — progress saved to', results_file)
                    break

                except Exception as e:
                    response, pred, correct = str(e)[:80], 'UNPARSEABLE', False
                    skipped += 1

                out_f.write(json.dumps({
                    'id':             inst['id'],
                    'dimension':      inst['dimension'],
                    'distance':       inst['distance'],
                    'context_tokens': inst['context_tokens'],
                    'gold':           inst['answer'],
                    'pred':           pred,
                    'correct':        correct,
                    'raw_response':   response[:120],
                }) + '\n')
                out_f.flush()

        del model
        gc.collect()
        torch.cuda.empty_cache()
        print(f'  ✅ {run_name} done. Unparseable: {unparseable}, Skipped: {skipped}')

    # Load full results
    df = pd.DataFrame([json.loads(l) for l in open(results_file)])
    acc = df['correct'].mean()
    print(f'\n  {run_name}  overall accuracy: {acc:.1%}  ({df["correct"].sum()}/{len(df)})')
    for dim, sub in df.groupby('dimension'):
        print(f'    {dim:<28}: {sub["correct"].mean():.1%}')
    return df


print('✅ run_model() ready.  Starting evaluations below...')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Model 1: Llama-3.2-3B  (~3.5 GB VRAM, ~20–40 min)
# ═══════════════════════════════════════════════════════════
df_llama3 = run_model(
    hf_id    = 'unsloth/Llama-3.2-3B-Instruct',
    run_name = 'llama3_3b',
)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Model 2: Qwen2.5-7B  (~5.5 GB VRAM, ~40–80 min)
# ═══════════════════════════════════════════════════════════
df_qwen = run_model(
    hf_id    = 'Qwen/Qwen2.5-7B-Instruct',
    run_name = 'qwen25_7b',
)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — Model 3: Mistral-7B  (~5.5 GB VRAM, ~40–80 min)
# ═══════════════════════════════════════════════════════════
df_mistral = run_model(
    hf_id    = 'mistralai/Mistral-7B-Instruct-v0.3',
    run_name = 'mistral_7b',
)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — Print results summary table
# ═══════════════════════════════════════════════════════════
DIMS = ['entity_consistency', 'temporal_coherence', 'causal_chain']
DIM_SHORT = {'entity_consistency': 'Entity', 'temporal_coherence': 'Temporal', 'causal_chain': 'Causal'}
MODEL_FILES = {'Llama-3.2-3B': 'llama3_3b', 'Qwen2.5-7B': 'qwen25_7b', 'Mistral-7B': 'mistral_7b'}

rows = []
for display, fname in MODEL_FILES.items():
    fpath = RESULTS_DIR / f'{fname}_results.jsonl'
    if not fpath.exists(): continue
    df = pd.DataFrame([json.loads(l) for l in open(fpath)])
    row = {'Model': display}
    for dim in DIMS:
        sub = df[df['dimension'] == dim]
        row[DIM_SHORT[dim]] = f"{sub['correct'].mean():.1%}" if len(sub) else 'N/A'
    row['Overall'] = f"{df['correct'].mean():.1%}"
    rows.append(row)

if rows:
    summary = pd.DataFrame(rows).set_index('Model')
    summary.to_csv(RESULTS_DIR / 'table3_accuracy_by_dimension.csv')
    print('\nTable 3 — Accuracy by Dimension')
    print('=' * 55)
    print(summary.to_string())
else:
    print('No results found yet — run at least one model cell above.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — Zip results for transfer
# ═══════════════════════════════════════════════════════════
import zipfile, shutil

zip_path = Path('cb_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(RESULTS_DIR.glob('*.jsonl')):
        zf.write(f, f.name)
        print(f'  Added: {f.name}  ({f.stat().st_size / 1024:.0f} KB)')
    for f in sorted(RESULTS_DIR.glob('*.csv')):
        zf.write(f, f.name)
        print(f'  Added: {f.name}')

size_kb = zip_path.stat().st_size / 1024
print(f'\n✅ Results zipped → {zip_path.resolve()}  ({size_kb:.0f} KB)')
print('Copy cb_results.zip back to your main machine and share with your assistant.')